# Factor Cross-Sectional Backtest

End-to-end validation of the factor long/short pipeline used by the
Factor Backtest frontend page.

**What this covers:**
- Load factors + prices from the same Parquet the API uses
- Run `create_signals_from_factor` for `mom_12_1`, `beta_60d`, `vol_60d`
- Diagnose per-date valid-stock counts (`min_stocks` drops)
- Run `calculate_portfolio_returns` with monthly and quarterly rebalancing
- Plot equity curves, drawdown, rolling Sharpe
- Compare factor-specific outlier bounds (`_infer_abs_bound`)

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
%matplotlib inline

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)

CWD = Path.cwd().resolve()
for root in [CWD, CWD.parent]:
    if (root / "config" / "settings.py").exists():
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
        PROJECT_ROOT = root
        break
else:
    raise FileNotFoundError("Cannot find project root (config/settings.py missing)")

DATA_DIR = PROJECT_ROOT / "data" / "factors"
assert DATA_DIR.exists(), f"Data directory missing: {DATA_DIR}"
print(f"Project root : {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

## 1. Load data (same artifacts the API uses)

In [ ]:
factors = pd.read_parquet(DATA_DIR / "factors_price.parquet")
prices = pd.read_parquet(DATA_DIR / "prices.parquet")

print(f"factors shape: {factors.shape}  index names: {factors.index.names}")
print(f"prices  shape: {prices.shape}   date range: {prices.index[0]} .. {prices.index[-1]}")
print(f"Factor columns: {list(factors.columns)}")

## 2. Inspect factor distributions

In [ ]:
FACTORS_TO_TEST = ["mom_12_1", "beta_60d", "vol_60d"]
FACTORS_TO_TEST = [f for f in FACTORS_TO_TEST if f in factors.columns]

for col in FACTORS_TO_TEST:
    s = factors[col].dropna()
    print(f"\n--- {col} ---")
    print(f"  count:  {len(s):,}")
    print(f"  mean:   {s.mean():.4f}")
    print(f"  median: {s.median():.4f}")
    print(f"  std:    {s.std():.4f}")
    print(f"  min:    {s.min():.4f}  max: {s.max():.4f}")
    print(f"  |val| >= 10: {(s.abs() >= 10).sum()} rows")
    print(f"  inf/nan:     {(~np.isfinite(s)).sum()} rows")

## 3. Per-date valid-stock counts

This is the key diagnostic: how many stocks have valid factor values
on each date after applying the outlier filter?  Dates with fewer than
`min_stocks` (default 20) produce **zero signals** and lead to the
flat-line periods visible in the frontend.

In [ ]:
from core.backtest.portfolio import _infer_abs_bound

MIN_STOCKS = 20

fig, axes = plt.subplots(len(FACTORS_TO_TEST), 1, figsize=(14, 4 * len(FACTORS_TO_TEST)), sharex=True)
if len(FACTORS_TO_TEST) == 1:
    axes = [axes]

for ax, col in zip(axes, FACTORS_TO_TEST):
    bound = _infer_abs_bound(col)
    mask = factors[col].notna() & np.isfinite(factors[col])
    if np.isfinite(bound):
        mask = mask & (factors[col].abs() < bound)
    valid_counts = factors.loc[mask].groupby(level="date").size()

    ax.plot(valid_counts.index, valid_counts.values, linewidth=0.6)
    ax.axhline(MIN_STOCKS, color="red", linestyle="--", linewidth=1, label=f"min_stocks={MIN_STOCKS}")
    below = valid_counts[valid_counts < MIN_STOCKS]
    ax.scatter(below.index, below.values, color="red", s=8, zorder=3)
    ax.set_ylabel("Valid stocks")
    ax.set_title(f"{col}  (bound={bound})  — {len(below)} dates below threshold")
    ax.legend(loc="lower right")

fig.tight_layout()
plt.show()

## 4. Generate signals and run backtests

In [ ]:
from core.backtest.portfolio import (
    calculate_portfolio_returns,
    create_signals_from_factor,
    sp500_universe_filter,
)
from core.metrics.performance import calculate_performance_metrics

uf = sp500_universe_filter()

REBAL_FREQS = ["ME", "QE"]
results = {}

for col in FACTORS_TO_TEST:
    for freq in REBAL_FREQS:
        key = f"{col} / {freq}"
        print(f"Running {key} ...")
        signals = create_signals_from_factor(
            factors, col,
            top_pct=0.20, bottom_pct=0.20,
            min_stocks=MIN_STOCKS,
            universe_filter=uf,
        )
        port = calculate_portfolio_returns(
            signals, prices,
            rebalance_freq=freq,
            transaction_cost=0.001,
        )
        results[key] = port

print(f"\nDone: {len(results)} backtests")

## 5. Performance summary table

In [ ]:
rows = []
for key, port in results.items():
    net = port["net_return"]
    m = calculate_performance_metrics(net)
    rows.append({"strategy": key, **m})

summary = pd.DataFrame(rows).set_index("strategy")
display(summary.round(4))

## 6. Equity curves

In [ ]:
from core.metrics.performance import calculate_cumulative_returns

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for key, port in results.items():
    cum = calculate_cumulative_returns(port["net_return"])
    axes[0].plot(cum.index, cum.values, label=key, linewidth=1)

axes[0].set_ylabel("Cumulative Return")
axes[0].set_title("Factor Cross-Section Equity Curves")
axes[0].legend(fontsize=8)

for key, port in results.items():
    net = port["net_return"]
    cum = (1 + net).cumprod()
    running_max = cum.cummax()
    dd = (cum - running_max) / running_max
    axes[1].plot(dd.index, dd.values * 100, label=key, linewidth=0.8)

axes[1].set_ylabel("Drawdown (%)")
axes[1].set_title("Drawdown")
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.show()

## 7. Position counts over time

Flat-line periods in the equity curve correspond to dates where
`n_long` + `n_short` = 0.

In [ ]:
fig, axes = plt.subplots(len(results), 1, figsize=(14, 3 * len(results)), sharex=True)
if len(results) == 1:
    axes = [axes]

for ax, (key, port) in zip(axes, results.items()):
    total_pos = port["n_long"] + port["n_short"]
    ax.fill_between(total_pos.index, total_pos.values, alpha=0.5)
    zero_days = (total_pos == 0).sum()
    ax.set_ylabel("# positions")
    ax.set_title(f"{key} — zero-position days: {zero_days}")

fig.tight_layout()
plt.show()

## 8. Factor-specific bounds comparison

Compare valid-count change between old `abs < 10` and new
`_infer_abs_bound` for `beta_60d`.

In [ ]:
col = "beta_60d" if "beta_60d" in factors.columns else FACTORS_TO_TEST[0]

base = factors[col].notna() & np.isfinite(factors[col])
old_mask = base & (factors[col].abs() < 10)
new_bound = _infer_abs_bound(col)
new_mask = base & (factors[col].abs() < new_bound) if np.isfinite(new_bound) else base

old_counts = factors.loc[old_mask].groupby(level="date").size()
new_counts = factors.loc[new_mask].groupby(level="date").size()

diff = (new_counts - old_counts).dropna()
print(f"Factor: {col}")
print(f"Old bound abs<10  -> dates below min_stocks: {(old_counts < MIN_STOCKS).sum()}")
print(f"New bound abs<{new_bound} -> dates below min_stocks: {(new_counts < MIN_STOCKS).sum()}")
print(f"Extra rows recovered (non-zero diff): {(diff > 0).sum()} dates")
print(f"Max extra stocks on a single date: {diff.max():.0f}")